<a href="https://colab.research.google.com/github/cpython-projects/da_27_07_2026/blob/main/lesson_22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Вступ до реляційних баз даних

## 0. Підготовка

Працюємо з **SQLite**: це СУБД, яка зберігає всю базу в одному файлі й уже вбудована в Python (модуль `sqlite3`). Нічого встановлювати не треба.

Щоб зручно дивитися результати, зробимо дві допоміжні функції:
- `sql(...)` — виконує `SELECT` і повертає звичний `DataFrame`;
- `run(...)` — виконує запит, що змінює базу, і фіксує зміни (`commit`);
- `try_sql(...)` — те саме, що `run`, але показує помилку замість падіння (для демонстрацій).

In [1]:
import os
import sqlite3
import pandas as pd

DB_NAME = 'school.db'

# починаємо з чистої бази при кожному запуску ноутбука
if os.path.exists(DB_NAME):
    os.remove(DB_NAME)

conn = sqlite3.connect(DB_NAME)   # підключення до бази (файл створиться автоматично)
cursor = conn.cursor()            # курсор — "вказівник", через який виконуємо запити

print('Версія SQLite:', sqlite3.sqlite_version)

Версія SQLite: 3.45.1


In [2]:
def sql(query, params=()):
    """Виконати SELECT і повернути результат як DataFrame."""
    return pd.read_sql_query(query, conn, params=params)


def run(query, params=()):
    """Виконати запит, що змінює дані/структуру, і зафіксувати зміни."""
    cursor.execute(query, params)
    conn.commit()
    # скільки рядків змінено (для INSERT/UPDATE/DELETE); для CREATE/DROP нічого не повертаємо
    return cursor.rowcount if cursor.rowcount >= 0 else None


def try_sql(query, params=()):
    """Як run(), але замість падіння показує помилку. Для демонстрацій."""
    try:
        n = run(query, params)
        print('✅ Виконано.' + (f' Змінено рядків: {n}' if n is not None else ''))
    except sqlite3.Error as e:
        conn.rollback()
        print(f'❌ {type(e).__name__}: {e}')

## 1. Базові поняття

**База даних (БД)** — структурований набір даних, що зберігається довго і з яким працюють багато користувачів/програм одночасно.

**СУБД (система управління базами даних)** — програма, яка зберігає БД, виконує запити, перевіряє правила цілісності, керує доступом. Приклади:

| Тип | Приклади | Де зустрінете |
|---|---|---|
| Реляційні (OLTP) | PostgreSQL, MySQL, MS SQL Server, Oracle, **SQLite** | бекенд продукту: користувачі, замовлення, оплати |
| Аналітичні сховища (OLAP / DWH) | ClickHouse, BigQuery, Snowflake, Redshift | продуктова аналітика: події, мільярди рядків |
| Нереляційні (NoSQL) | MongoDB, Redis, Cassandra | кеші, документи, специфічні задачі |

Майже всі вони розуміють **SQL** (Structured Query Language) — декларативну мову запитів: ми описуємо *що* хочемо отримати, а СУБД сама вирішує *як*.

### Чому не просто Excel / CSV?
- **Обсяг**: мільйони й мільярди рядків
- **Одночасна робота** багатьох користувачів і програм
- **Цілісність**: база сама не дасть записати некоректні дані (дублікати, порожні обов'язкові поля, посилання «в нікуди»)
- **Транзакції**: або виконалось усе, або нічого
- **Права доступу**, резервні копії, швидкий пошук (індекси)

### Реляційна модель
Дані зберігаються у вигляді **таблиць** (математично — *відношень*, relation, звідси й назва), між якими є **зв'язки**.

| Термін БД | Синоніми | Аналог у pandas |
|---|---|---|
| Таблиця | відношення, relation | `DataFrame` |
| Рядок | запис, кортеж, row, record | рядок `DataFrame` |
| Стовпець | атрибут, поле, column | `Series` / колонка |
| Тип стовпця | домен | `dtype` |
| Первинний ключ | primary key | (приблизно) унікальний `index` |
| Схема | структура таблиць і зв'язків | — |

Головна відмінність від `DataFrame`: у таблиці БД **структура задається заздалегідь** — назви стовпців, типи, правила. Дані, що не відповідають правилам, база просто не прийме.

### Категорії SQL-команд
| Категорія | Для чого | Команди |
|---|---|---|
| **DDL** (Data Definition) | структура | `CREATE`, `ALTER`, `DROP` |
| **DML** (Data Manipulation) | дані | `INSERT`, `UPDATE`, `DELETE` |
| **DQL** (Data Query) | читання | `SELECT` |
| **TCL** (Transaction Control) | транзакції | `COMMIT`, `ROLLBACK` |
| **DCL** (Data Control) | права доступу | `GRANT`, `REVOKE` |

Аналітик найчастіше пише `SELECT`, але щоб писати його правильно, треба розуміти, **як і чому** влаштовані таблиці. Цим і займаємось сьогодні.

## 2. Проблеми зберігання всього в одній таблиці

Уявімо, що адміністратор онлайн-школи веде облік «як в Excel» — одна велика таблиця, де кожен рядок = «студент навчається в групі».

In [3]:
run("""
CREATE TABLE school_flat (
    student_name   TEXT,
    student_email  TEXT,
    student_phones TEXT,
    group_title    TEXT,
    course_title   TEXT,
    course_hours   INTEGER,
    teacher_name   TEXT,
    teacher_email  TEXT
)
""")

In [4]:
flat_rows = [
    ('Олена Коваль',   'olena@gmail.com',  '050-111-11-11, 067-222-22-22', 'DA-01', 'Аналітика даних',  120, 'Петро Іваненко', 'petro@school.ua'),
    ('Олена Коваль',   'olena@gmail.com',  '050-111-11-11, 067-222-22-22', 'PY-01', 'Python',            80, 'Марія Литвин',   'maria@school.ua'),
    ('Андрій Мельник', 'andrii@gmail.com', '063-333-33-33',                'DA-01', 'Аналітика даних',  120, 'Петро Іваненко', 'petro@school.ua'),
    ('Ірина Шевченко', 'iryna@gmail.com',  '050-444-44-44, 093-555-55-55', 'DA-02', 'Аналітика даних',  120, 'Петро Іваненко', 'petro@school.ua'),
    ('Ірина Шевченко', 'iryna@gmail.com',  '050-444-44-44, 093-555-55-55', 'PY-01', 'Python',            80, 'Марія Литвин',   'maria@school.ua'),
    ('Максим Бондар',  'maksym@gmail.com', '066-666-66-66',                'DA-02', 'Аналітика даних',  120, 'Петро Іваненко', 'petro@school.ua'),
    ('Софія Ткаченко', 'sofia@gmail.com',  '097-777-77-77',                'ML-01', 'Machine Learning', 100, 'Олег Савчук',    'oleg@school.ua'),
]

cursor.executemany("INSERT INTO school_flat VALUES (?, ?, ?, ?, ?, ?, ?, ?)", flat_rows)
conn.commit()

sql("SELECT * FROM school_flat")

,student_name,student_email,student_phones,group_title,course_title,course_hours,teacher_name,teacher_email
0,Олена Коваль,olena@gmail.com,"050-111-11-11, 067-222-22-22",DA-01,Аналітика даних,120,Петро Іваненко,petro@school.ua
1,Олена Коваль,olena@gmail.com,"050-111-11-11, 067-222-22-22",PY-01,Python,80,Марія Литвин,maria@school.ua
2,Андрій Мельник,andrii@gmail.com,063-333-33-33,DA-01,Аналітика даних,120,Петро Іваненко,petro@school.ua
3,Ірина Шевченко,iryna@gmail.com,"050-444-44-44, 093-555-55-55",DA-02,Аналітика даних,120,Петро Іваненко,petro@school.ua
4,Ірина Шевченко,iryna@gmail.com,"050-444-44-44, 093-555-55-55",PY-01,Python,80,Марія Литвин,maria@school.ua
5,Максим Бондар,maksym@gmail.com,066-666-66-66,DA-02,Аналітика даних,120,Петро Іваненко,petro@school.ua
6,Софія Ткаченко,sofia@gmail.com,097-777-77-77,ML-01,Machine Learning,100,Олег Савчук,oleg@school.ua


На перший погляд — зручно: все в одному місці. Тепер подивимось, що з цим не так. Будемо розбирати проблеми по одній.

### Проблема 1. Дублювання даних (надлишковість)
Скільки разів ми записали одне й те саме про викладачів і курси?

In [5]:
flat = sql("SELECT * FROM school_flat")
flat[['teacher_name', 'teacher_email']].value_counts()

,,count
teacher_name,teacher_email,
Петро Іваненко,petro@school.ua,4
Марія Литвин,maria@school.ua,2
Олег Савчук,oleg@school.ua,1


Email Петра записаний **4 рази**, назва курсу «Аналітика даних» і його тривалість — теж 4 рази, телефони Олени та Ірини — по 2 рази.

Наслідки: зайве місце, а головне — **кожна копія може «роз'їхатись»** з іншими.

### Проблема 2. Кілька значень в одній клітинці
Знайдемо студента за номером телефону `067-222-22-22`:

In [6]:
sql("SELECT student_name FROM school_flat WHERE student_phones = '067-222-22-22'")

,student_name


Порожньо! Хоча такий номер є — він «склеєний» з іншим у рядку `'050-111-11-11, 067-222-22-22'`.

Можна шукати по шаблону (`LIKE '%067-222%'`), але це крихко: хтось запише `0672222222` або `+380 67 ...`, і пошук зламається. Також не можна нормально порахувати, скільки телефонів у студента, чи перевірити, що номер унікальний.

In [7]:
sql("SELECT DISTINCT student_name FROM school_flat WHERE student_phones LIKE '%067-222%'")

,student_name
0,Олена Коваль


### Проблема 3. Аномалія оновлення (update anomaly)
Петро Іваненко змінив email. Адміністратор оновив його... але тільки в групі DA-01, про DA-02 забув.

In [8]:
run("""
UPDATE school_flat
SET teacher_email = 'p.ivanenko@school.ua'
WHERE group_title = 'DA-01'
""")

sql("SELECT DISTINCT teacher_name, teacher_email FROM school_flat ORDER BY teacher_name")

,teacher_name,teacher_email
0,Марія Литвин,maria@school.ua
1,Олег Савчук,oleg@school.ua
2,Петро Іваненко,p.ivanenko@school.ua
3,Петро Іваненко,petro@school.ua


Тепер у Петра **два різні email**. Який правильний? База не знає. Дані стали **суперечливими**.

### Проблема 4. Аномалія вставки (insert anomaly)
Школа запускає новий курс «SQL для аналітиків» (60 годин). Груп і студентів ще немає. Як його записати?

In [9]:
run("""
INSERT INTO school_flat (course_title, course_hours)
VALUES ('SQL для аналітиків', 60)
""")

sql("SELECT * FROM school_flat")

,student_name,student_email,student_phones,group_title,course_title,course_hours,teacher_name,teacher_email
0,Олена Коваль,olena@gmail.com,"050-111-11-11, 067-222-22-22",DA-01,Аналітика даних,120,Петро Іваненко,p.ivanenko@school.ua
1,Олена Коваль,olena@gmail.com,"050-111-11-11, 067-222-22-22",PY-01,Python,80,Марія Литвин,maria@school.ua
2,Андрій Мельник,andrii@gmail.com,063-333-33-33,DA-01,Аналітика даних,120,Петро Іваненко,p.ivanenko@school.ua
3,Ірина Шевченко,iryna@gmail.com,"050-444-44-44, 093-555-55-55",DA-02,Аналітика даних,120,Петро Іваненко,petro@school.ua
4,Ірина Шевченко,iryna@gmail.com,"050-444-44-44, 093-555-55-55",PY-01,Python,80,Марія Литвин,maria@school.ua
5,Максим Бондар,maksym@gmail.com,066-666-66-66,DA-02,Аналітика даних,120,Петро Іваненко,petro@school.ua
6,Софія Ткаченко,sofia@gmail.com,097-777-77-77,ML-01,Machine Learning,100,Олег Савчук,oleg@school.ua
7,None,None,None,None,SQL для аналітиків,60,None,None


Довелося створити «рядок-привид», де студент, група і викладач порожні. Тепер кожен, хто рахує студентів чи групи, мусить пам'ятати про такі рядки. Інакше кажучи: **неможливо зберегти факт про одну сутність (курс), не маючи даних про іншу (студента)**.

In [10]:
# приберемо "привида"
run("DELETE FROM school_flat WHERE student_name IS NULL")

1

### Проблема 5. Аномалія видалення (delete anomaly)
Софія — єдина студентка курсу Machine Learning — вирішила піти. Видаляємо її:

In [11]:
run("DELETE FROM school_flat WHERE student_email = 'sofia@gmail.com'")

sql("SELECT DISTINCT course_title, teacher_name FROM school_flat")

,course_title,teacher_name
0,Аналітика даних,Петро Іваненко
1,Python,Марія Литвин


Разом зі студенткою **зник курс Machine Learning і викладач Олег Савчук**. Ми хотіли видалити один факт («Софія навчається»), а втратили ще два («такий курс існує», «такий викладач працює»).

### Висновок
Усі проблеми мають одну причину: **одна таблиця описує кілька різних сутностей** — студентів, їхні телефони, групи, курси, викладачів і факт «студент навчається в групі».

> **Правило:** одна таблиця — одна сутність. Кожен факт зберігається **в одному місці**.

Формальний спосіб прийти до цього — **нормалізація**.

In [12]:
# плоска таблиця нам більше не потрібна
run("DROP TABLE school_flat")

## 3. Нормалізація на простих прикладах

**Нормалізація** — це поетапне розбиття таблиці на кілька менших, щоб прибрати дублювання та аномалії з розділу 2. Кожен етап називається **нормальною формою (НФ)**, і кожна наступна форма включає вимоги попередньої.

Перш ніж повертатися до таблиці школи, розберемо кожну форму на маленькій таблиці з 3–4 рядків.

### 3.0. Інтуїція
Два правила, з яких випливає все інше:
1. **Одна таблиця — про одну «річ»** (студентів, курси, оцінки...).
2. **Кожен факт записаний один раз.**

Ще одне поняття: **залежність**. Кажемо «*B залежить від A*», якщо, знаючи A, ми точно знаємо B.
- Знаючи номер паспорта, точно знаємо ПІБ → ПІБ залежить від номера паспорта.
- Знаючи ПІБ, номер паспорта **не** знаємо (бувають тезки) → номер паспорта від ПІБ не залежить.

### 3.1. Перша нормальна форма (1НФ): в одній клітинці — одне значення

Таблиця студентів, де телефони записані через кому:

In [13]:
phones_bad = pd.DataFrame({
    'студент':  ['Олена', 'Андрій', 'Ірина'],
    'телефони': ['050-111, 067-222', '063-333', '050-444, 093-555'],
})
phones_bad

,студент,телефони
0,Олена,"050-111, 067-222"
1,Андрій,063-333
2,Ірина,"050-444, 093-555"


Що не так: не можна точно знайти студента за номером, порахувати телефони чи перевірити, що номер не повторюється.

Інша поширена помилка — повторювані стовпці:

| студент | телефон_1 | телефон_2 | телефон_3 |
|---|---|---|---|
| Олена | 050-111 | 067-222 | |
| Андрій | 063-333 | | |

Скільки таких стовпців робити? А коли у когось з'явиться четвертий номер, доведеться міняти структуру таблиці.

**Рішення:** кожен телефон — окремий рядок в **окремій таблиці** «студент — телефон»:

In [14]:
phones_good = (
    phones_bad
    .assign(телефон=phones_bad['телефони'].str.split(', '))
    .explode('телефон')
    .drop(columns='телефони')
    .reset_index(drop=True)
)
phones_good

,студент,телефон
0,Олена,050-111
1,Олена,067-222
2,Андрій,063-333
3,Ірина,050-444
4,Ірина,093-555


Чому окрема таблиця, а не просто більше рядків у таблиці студентів? Якби там були ще email, дата народження тощо, вони б повторювались у кожному рядку з телефоном. Це те дублювання, з яким ми боремося.

> **1НФ:** у кожній клітинці одне значення, немає стовпців `x_1, x_2, x_3`.

### 3.2. Ключ: що однозначно визначає рядок?

Журнал оцінок:

In [15]:
grades = pd.DataFrame({
    'студент': ['Олена', 'Олена', 'Андрій', 'Андрій'],
    'предмет': ['SQL', 'Python', 'SQL', 'Python'],
    'оцінка':  [95, 88, 76, 90],
})
grades

,студент,предмет,оцінка
0,Олена,SQL,95
1,Олена,Python,88
2,Андрій,SQL,76
3,Андрій,Python,90


Чи можна знайти потрібний рядок, знаючи лише студента? Ні: в Олени два рядки. А лише предмет? Теж ні. Перевіримо, чи є повтори:

In [16]:
print('Повтори за студентом:          ', grades.duplicated(['студент']).any())
print('Повтори за предметом:          ', grades.duplicated(['предмет']).any())
print('Повтори за (студент, предмет): ', grades.duplicated(['студент', 'предмет']).any())

Повтори за студентом:           True
Повтори за предметом:           True
Повтори за (студент, предмет):  False


Рядок однозначно визначає лише **пара (студент, предмет)**. Це і є **ключ** таблиці. Ключ з кількох стовпців називають **складеним**.

> **Ключ** — мінімальний набір стовпців, значення якого не повторюються і за яким можна знайти рівно один рядок.

Від ключа відштовхуються наступні дві нормальні форми: для кожного стовпця питаємо, **від чого саме** він залежить.

### 3.3. Друга нормальна форма (2НФ): стовпець залежить від усього ключа

Додамо в журнал місто студента:

In [17]:
grades_city = pd.DataFrame({
    'студент': ['Олена', 'Олена', 'Андрій', 'Андрій'],
    'предмет': ['SQL', 'Python', 'SQL', 'Python'],
    'оцінка':  [95, 88, 76, 90],
    'місто':   ['Київ', 'Київ', 'Львів', 'Львів'],
})
grades_city

,студент,предмет,оцінка,місто
0,Олена,SQL,95,Київ
1,Олена,Python,88,Київ
2,Андрій,SQL,76,Львів
3,Андрій,Python,90,Львів


Ключ — пара **(студент, предмет)**. Для кожного неключового стовпця питаємо: *він залежить від усієї пари чи лише від однієї частини?*

**Оцінка:**
- один студент, різні предмети → оцінки різні (Олена: SQL 95, Python 88);
- один предмет, різні студенти → оцінки різні (SQL: Олена 95, Андрій 76).

Оцінку визначає **вся пара**. ✅ Стовпець на своєму місці.

**Місто:**
- один студент, різні предмети → місто те саме (Олена: Київ, Київ);
- тобто предмет на місто не впливає.

Місто визначає **лише студент**, тобто лише частина ключа. ❌ Тому воно і повторюється в кожному рядку студента.

Перевіримо це кодом: скільки різних значень має кожен стовпець **для одного студента**?

In [18]:
grades_city.groupby('студент')[['оцінка', 'місто']].nunique()

,оцінка,місто
студент,,
Андрій,2,1
Олена,2,1


Для одного студента місто завжди одне, а оцінок кілька. Отже, місто залежить лише від студента.

Чим це погано: Олена переїхала до Львова, і місто треба оновлювати **в кожному** її рядку. Пропустимо один, і дані суперечать самі собі (аномалія оновлення з розділу 2).

**Рішення:** місто виносимо туди, де ключ — сам студент:

In [19]:
students_simple = grades_city[['студент', 'місто']].drop_duplicates().reset_index(drop=True)
students_simple

,студент,місто
0,Олена,Київ
1,Андрій,Львів


In [20]:
grades_simple = grades_city[['студент', 'предмет', 'оцінка']]
grades_simple

,студент,предмет,оцінка
0,Олена,SQL,95
1,Олена,Python,88
2,Андрій,SQL,76
3,Андрій,Python,90


> **2НФ:** таблиця в 1НФ, і кожен неключовий стовпець залежить від **усього** ключа, а не від його частини.
>
> Якщо ключ складається з одного стовпця, таблиця в 1НФ автоматично в 2НФ.

### 3.4. Третя нормальна форма (3НФ): стовпець описує сам рядок, а не інший стовпець

Таблиця студентів із групою та куратором групи:

In [21]:
students_groups = pd.DataFrame({
    'студент': ['Олена', 'Андрій', 'Ірина', 'Максим'],
    'група':   ['DA-01', 'DA-01', 'PY-01', 'PY-01'],
    'куратор': ['Петро', 'Петро', 'Марія', 'Марія'],
})
students_groups

,студент,група,куратор
0,Олена,DA-01,Петро
1,Андрій,DA-01,Петро
2,Ірина,PY-01,Марія
3,Максим,PY-01,Марія


Ключ — **студент** (один стовпець, отже 2НФ виконується автоматично). Тепер питаємо: *кожен стовпець описує студента чи щось інше?*

**Група:** різні студенти можуть бути в різних групах, і кожен студент у своїй групі. Це властивість студента. ✅

**Куратор:**
- *куратор змінюється, якщо поміняти студента, але залишити ту саму групу?* Ні: Олена й Андрій обидва в DA-01, і в обох куратор Петро.

Куратора визначає **група**, а не студент. ❌ Ланцюжок такий: студент → група → куратор. Таку залежність «через посередника» називають **транзитивною**.

Перевіримо: скільки різних кураторів буває в одній групі?

In [22]:
students_groups.groupby('група')['куратор'].nunique()

,куратор
група,
DA-01,1
PY-01,1


У кожної групи рівно один куратор, тобто куратор — властивість групи.

Чим це погано: група DA-01 отримала нового куратора, і його треба оновити в рядку **кожного** студента групи.

**Рішення:** виносимо дані про групу в окрему таблицю:

In [23]:
groups_simple = students_groups[['група', 'куратор']].drop_duplicates().reset_index(drop=True)
groups_simple

,група,куратор
0,DA-01,Петро
1,PY-01,Марія


In [24]:
students_simple_3nf = students_groups[['студент', 'група']]
students_simple_3nf

,студент,група
0,Олена,DA-01
1,Андрій,DA-01
2,Ірина,PY-01
3,Максим,PY-01


> **3НФ:** таблиця в 2НФ, і кожен неключовий стовпець описує **сам рядок** (залежить від ключа), а не інший неключовий стовпець.

### 3.5. Шпаргалка: три питання до таблиці

| # | Питання | Якщо «так» | Що робимо |
|---|---|---|---|
| 1 | У клітинці список (або є стовпці `x_1, x_2, ...`)? | порушена **1НФ** | виносимо значення в окрему таблицю, по одному в рядку |
| 2 | Стовпець залежить лише від **частини** складеного ключа? | порушена **2НФ** | виносимо його в таблицю, де ключ — ця частина |
| 3 | Стовпець описує **інший неключовий стовпець**, а не сам рядок? | порушена **3НФ** | виносимо його в таблицю, де ключ — цей інший стовпець |

Ключова фраза, яку варто запам'ятати: кожен неключовий стовпець залежить *«від ключа, від усього ключа і ні від чого, крім ключа»*.

### Важливо для аналітика
- Існують і вищі форми (НФБК/BCNF, 4НФ, 5НФ), але на практиці зазвичай достатньо **3НФ**.
- **Денормалізація** — свідоме об'єднання таблиць назад. В аналітичних сховищах (DWH) часто роблять «широкі» таблиці, бо там дані переважно читають, а не змінюють. Але це **рішення**, а не випадковість.
- Продуктові бази нормалізовані, тому аналітик постійно **з'єднує** таблиці назад (`JOIN`). JOIN розберемо в наступних уроках.

Таблицю школи з розділу 2 ми перепроєктуємо за цією шпаргалкою в розділі 6, коли вже знатимемо, як робити ключі та зв'язки в SQL.

## 4. Ключі та зв'язки

### Первинний ключ (PRIMARY KEY, PK)
Стовпець (або набір стовпців), що **однозначно ідентифікує рядок**.
- значення **унікальні**;
- значення **не NULL**;
- у таблиці **один** первинний ключ (але він може складатися з кількох стовпців — **складений ключ**).

**Природний ключ** — має сенс у реальному світі: email, номер паспорта, ІПН, назва групи.
**Сурогатний ключ** — штучний номер `id`, який не несе змісту.

У простих прикладах розділу 3 ключем було ім'я студента. Але тезки трапляються, тому ім'я — поганий ключ. А email? Він унікальний, але теж не найкращий вибір:
- email **змінюється** (як у Петра) — тоді довелося б міняти його в усіх таблицях, що на нього посилаються;
- email — персональні дані, не хочеться копіювати їх по всій базі;
- порівнювати цілі числа швидше, ніж рядки.

Тому стандартна практика: **сурогатний `id` як PK + `UNIQUE` на природному ключі** (email).

Особливості SQLite:
- `id INTEGER PRIMARY KEY` — це псевдонім вбудованого `rowid`; якщо не передати значення, SQLite сам видасть наступний номер.
- `AUTOINCREMENT` додатково гарантує, що номери видалених рядків **не будуть використані повторно**. Зазвичай це не потрібно, а працює трохи повільніше.

### Зовнішній ключ (FOREIGN KEY, FK)
Стовпець, значення якого **посилається на первинний ключ іншої таблиці**. Він гарантує **посилальну цілісність**: не можна послатися на те, чого не існує.

> ⚠️ **Пастка SQLite:** за замовчуванням зовнішні ключі **не перевіряються**! Їх треба увімкнути командою `PRAGMA foreign_keys = ON` — для **кожного нового підключення**. В PostgreSQL/MySQL вони працюють завжди.

Перевіримо на маленьких демо-таблицях:

In [25]:
run("""
CREATE TABLE demo_courses (
    id    INTEGER PRIMARY KEY,
    title TEXT NOT NULL
)
""")

run("""
CREATE TABLE demo_groups (
    id        INTEGER PRIMARY KEY,
    title     TEXT NOT NULL,
    course_id INTEGER NOT NULL REFERENCES demo_courses(id)
)
""")

run("INSERT INTO demo_courses (title) VALUES ('Python')")
sql("SELECT * FROM demo_courses")

,id,title
0,1,Python


In [26]:
# курсу з id = 999 не існує, але...
try_sql("INSERT INTO demo_groups (title, course_id) VALUES ('Група-привид', 999)")
sql("SELECT * FROM demo_groups")

✅ Виконано. Змінено рядків: 1


,id,title,course_id
0,1,Група-привид,999


SQLite мовчки записав посилання «в нікуди». Вмикаємо перевірку:

In [27]:
cursor.execute("PRAGMA foreign_keys = ON")
sql("PRAGMA foreign_keys")   # 1 = увімкнено

,foreign_keys
0,1


In [28]:
run("DELETE FROM demo_groups")   # прибираємо "привида"

try_sql("INSERT INTO demo_groups (title, course_id) VALUES ('Група-привид', 999)")
try_sql("INSERT INTO demo_groups (title, course_id) VALUES ('PY-01', 1)")

❌ IntegrityError: FOREIGN KEY constraint failed
✅ Виконано. Змінено рядків: 1


### Що робити при видаленні батьківського рядка? `ON DELETE`
Що станеться з групами, якщо видалити курс?

| Політика | Поведінка | Коли доречно |
|---|---|---|
| `NO ACTION` / `RESTRICT` (за замовчуванням) | заборонити видалення, поки є посилання | курс, у якого є групи |
| `CASCADE` | видалити й залежні рядки | телефони студента, записи студента в групи |
| `SET NULL` | у залежних рядках поставити `NULL` | викладач звільнився — група лишається без викладача |
| `SET DEFAULT` | поставити значення за замовчуванням | рідко |

Є аналогічне `ON UPDATE`, але з сурогатними `id` воно майже не потрібне.

In [29]:
# у курсу є група -> видалити не дасть (поведінка за замовчуванням)
try_sql("DELETE FROM demo_courses WHERE id = 1")

❌ IntegrityError: FOREIGN KEY constraint failed


### Таблиця, що посилається сама на себе (self-reference)
Зовнішній ключ може посилатися на **ту саму таблицю**. Так зберігають ієрархії:
- у школі: досвідчений викладач — **ментор** для нових (`teachers.mentor_id → teachers.id`);
- співробітник → керівник (`employees.manager_id`);
- категорія товару → батьківська категорія (`categories.parent_id`);
- коментар → коментар, на який відповідають (`comments.reply_to_id`);
- **продуктова аналітика**: користувач → хто його запросив (`users.referrer_id`) — реферальна програма.

У корені ієрархії значення `NULL` (у головного ментора немає ментора).

In [30]:
run("""
CREATE TABLE demo_teachers (
    id        INTEGER PRIMARY KEY,
    name      TEXT NOT NULL,
    mentor_id INTEGER REFERENCES demo_teachers(id) ON DELETE SET NULL
)
""")

cursor.executemany(
    "INSERT INTO demo_teachers (id, name, mentor_id) VALUES (?, ?, ?)",
    [
        (1, 'Петро Іваненко', None),  # головний ментор
        (2, 'Марія Литвин',   1),
        (3, 'Олег Савчук',    1),
        (4, 'Анна Гнатюк',    2),     # підопічна Марії
    ],
)
conn.commit()

sql("SELECT * FROM demo_teachers")

,id,name,mentor_id
0,1,Петро Іваненко,NaN
1,2,Марія Литвин,1.0
2,3,Олег Савчук,1.0
3,4,Анна Гнатюк,2.0


Чому `mentor_id` показується як `1.0` і `NaN`? Це pandas: у цілочисельній колонці з'явився пропуск (`NULL` → `NaN`), а `NaN` буває лише у `float`. У самій базі значення цілі.

In [31]:
# підопічні Петра (id = 1)
sql("SELECT name FROM demo_teachers WHERE mentor_id = 1")

,name
0,Марія Литвин
1,Олег Савчук


In [32]:
# не можна вказати ментора, якого не існує
try_sql("INSERT INTO demo_teachers (name, mentor_id) VALUES ('Новенький', 42)")

❌ IntegrityError: FOREIGN KEY constraint failed


Щоб отримати таблицю «викладач — ім'я ментора», потрібно з'єднати таблицю **саму з собою** (self-JOIN). Це розберемо в уроці про JOIN.

### Типи зв'язків між таблицями
| Тип | Приклад | Як реалізується |
|---|---|---|
| **1 : 1** | студент — його профіль/паспортні дані | FK + `UNIQUE` (або спільний PK) |
| **1 : N** (один до багатьох) | курс → групи; студент → телефони | FK у таблиці «багатьох» (`study_groups.course_id`) |
| **M : N** (багато до багатьох) | студенти ↔ групи | **проміжна таблиця** (`students_in_groups`) з двома FK |
| **self** (на себе) | викладач → ментор | FK на ту саму таблицю |

Зв'язок M:N **неможливо** зробити напряму — тільки через проміжну таблицю.

In [33]:
# прибираємо демо-таблиці (спочатку дочірні!)
run("DROP TABLE demo_groups")
run("DROP TABLE demo_courses")
run("DROP TABLE demo_teachers")

## 5. Типи даних, значення за замовчуванням, NULL

### Типи даних у SQLite
SQLite має лише 5 **класів зберігання**:

| Клас | Що зберігає |
|---|---|
| `NULL` | відсутність значення |
| `INTEGER` | ціле число |
| `REAL` | число з плаваючою крапкою |
| `TEXT` | рядок |
| `BLOB` | двійкові дані «як є» (файли, картинки) |

Окремих типів для дати/часу та boolean **немає**.

Тип, вказаний у `CREATE TABLE`, — це лише **рекомендація** (*type affinity*). SQLite спробує привести значення, а якщо не вийде — збереже як є:
- назва містить `INT` → INTEGER; `CHAR`, `TEXT` → TEXT; `REAL`, `FLOA`, `DOUB` → REAL; інше (`DATE`, `NUMERIC`, `BOOLEAN`) → NUMERIC;
- `VARCHAR(5)` — обмеження довжини **ігнорується**.

Переконаємось за допомогою функції `typeof()`:

In [34]:
run("""
CREATE TABLE types_demo (
    i INTEGER,
    t TEXT,
    r REAL,
    v VARCHAR(5),
    d DATE
)
""")

run("INSERT INTO types_demo VALUES ('42',  42,  '3.14', 'дуже довгий рядок', '2026-11-04')")
run("INSERT INTO types_demo VALUES ('abc', 3.5, 10,     'ok',                 20261104)")

sql("""
SELECT i, typeof(i) AS type_i,
       t, typeof(t) AS type_t,
       r, typeof(r) AS type_r,
       v, typeof(v) AS type_v,
       d, typeof(d) AS type_d
FROM types_demo
""")

,i,type_i,t,type_t,r,type_r,v,type_v,d,type_d
0,42,integer,42,text,3.14,real,дуже довгий рядок,text,2026-11-04,text
1,abc,text,3.5,text,10.00,real,ok,text,20261104,integer


`'abc'` спокійно записався в INTEGER, а рядок із 17 символів — у `VARCHAR(5)`. В PostgreSQL обидва запити завершились би помилкою.

З версії 3.37 у SQLite є **STRICT-таблиці** — з жорсткою перевіркою типів (дозволені лише `INTEGER`, `REAL`, `TEXT`, `BLOB`, `ANY`):

In [35]:
run("CREATE TABLE types_strict (i INTEGER, t TEXT) STRICT")

try_sql("INSERT INTO types_strict VALUES ('42', 'ok')")    # '42' можна привести до числа
try_sql("INSERT INTO types_strict VALUES ('abc', 'ok')")   # а 'abc' — ні

✅ Виконано. Змінено рядків: 1
❌ IntegrityError: cannot store TEXT value in INTEGER column types_strict.i


### Порівняння з PostgreSQL (з ним ви, найімовірніше, зустрінетесь на роботі)
| Що зберігаємо | SQLite | PostgreSQL |
|---|---|---|
| Ціле число, id | `INTEGER` | `INTEGER`, `BIGINT`, `SERIAL` / `GENERATED AS IDENTITY` |
| Гроші | `INTEGER` (у копійках) або `NUMERIC` | `NUMERIC(10, 2)` |
| Дробове (метрики, координати) | `REAL` | `REAL`, `DOUBLE PRECISION` |
| Рядок | `TEXT` | `VARCHAR(n)`, `TEXT` |
| Так / ні | `INTEGER` 0/1 | `BOOLEAN` |
| Дата | `TEXT` `'YYYY-MM-DD'` | `DATE` |
| Дата й час | `TEXT` `'YYYY-MM-DD HH:MM:SS'` | `TIMESTAMP`, `TIMESTAMPTZ` |
| JSON | `TEXT` + функції `json_*` | `JSON`, `JSONB` |

#### Чому гроші не можна зберігати у `REAL`/`FLOAT`

In [36]:
sql("SELECT 0.1 + 0.2 = 0.3 AS is_equal, 0.1 + 0.2 - 0.3 AS difference")

,is_equal,difference
0,0,5.551115e-17


#### Дати: тільки формат ISO 8601 (`YYYY-MM-DD`)
Дати в SQLite — це рядки, і порівнюються вони **як рядки**. Формат `YYYY-MM-DD` сортується правильно, а звичний нам `ДД.ММ.РРРР` — ні:

In [37]:
sql("""
SELECT
    '2026-02-01' > '2025-11-15' AS iso_format,   -- 1 лютого 2026 пізніше за 15 листопада 2025? -> 1 (так)
    '01.02.2026' > '15.11.2025' AS dmy_format    -- те саме в форматі ДД.ММ.РРРР -> 0 (неправильно!)
""")

,iso_format,dmy_format
0,1,0


Для роботи з датами є вбудовані функції:

In [38]:
sql("""
SELECT
    date('now')                                AS today,
    datetime('now')                            AS now_utc,
    date('now', '-7 days')                     AS week_ago,
    date('now', 'start of month')              AS month_start,
    julianday('2026-11-04') - julianday('2026-09-01') AS days_between,
    date('2026-02-30')                         AS feb_30,      -- "перекочує" в березень
    date('вчора')                              AS not_a_date   -- не дата -> NULL
""")

,today,now_utc,week_ago,month_start,days_between,feb_30,not_a_date
0,2026-11-04,2026-11-04 15:20:42,2026-10-28,2026-11-01,64.0,2026-03-02,None


Тому перевіряти коректність дати зручно умовою `CHECK (x = date(x))`: для `'2026-02-30'` чи `'вчора'` рівність не виконається. Ми використаємо це в схемі школи.

### Обмеження (constraints) і значення за замовчуванням
| Обмеження | Що робить |
|---|---|
| `NOT NULL` | значення обов'язкове |
| `UNIQUE` | значення не повторюються |
| `DEFAULT ...` | значення, якщо його не передали (`0`, `'pending'`, `CURRENT_DATE`, `CURRENT_TIMESTAMP`) |
| `CHECK (умова)` | довільна перевірка |
| `PRIMARY KEY`, `REFERENCES` | розібрали вище |

Приклад з продуктового світу — таблиця оплат:

In [39]:
run("""
CREATE TABLE payments_demo (
    id          INTEGER PRIMARY KEY,
    amount_uah  NUMERIC NOT NULL CHECK (amount_uah > 0),
    status      TEXT    NOT NULL DEFAULT 'pending'
                        CHECK (status IN ('pending', 'paid', 'refunded')),
    is_test     INTEGER NOT NULL DEFAULT 0 CHECK (is_test IN (0, 1)),
    promo_code  TEXT,                                   -- необов'язкове поле
    created_at  TEXT    NOT NULL DEFAULT CURRENT_TIMESTAMP
)
""")

# передаємо тільки суму — решту заповнять DEFAULT-и
run("INSERT INTO payments_demo (amount_uah) VALUES (1500)")
sql("SELECT * FROM payments_demo")

,id,amount_uah,status,is_test,promo_code,created_at
0,1,1500,pending,0,None,2026-11-04 15:20:42


In [40]:
try_sql("INSERT INTO payments_demo (amount_uah) VALUES (-100)")                    # CHECK
try_sql("INSERT INTO payments_demo (amount_uah, status) VALUES (900, 'done')")     # CHECK
try_sql("INSERT INTO payments_demo (status) VALUES ('paid')")                      # NOT NULL
try_sql("INSERT INTO payments_demo (amount_uah, status) VALUES (900, NULL)")       # NOT NULL сильніший за DEFAULT

❌ IntegrityError: CHECK constraint failed: amount_uah > 0
❌ IntegrityError: CHECK constraint failed: status IN ('pending', 'paid', 'refunded')
❌ IntegrityError: NOT NULL constraint failed: payments_demo.amount_uah
❌ IntegrityError: NOT NULL constraint failed: payments_demo.status


Зверніть увагу на останній приклад: `DEFAULT` спрацьовує, тільки якщо стовпець **не вказали** в `INSERT`. Явно переданий `NULL` — це теж значення.

### NULL — «значення невідоме / відсутнє»
`NULL` — це **не 0, не порожній рядок і не `False`**. Це «нічого не відомо». Звідси незвична логіка: будь-яке порівняння з `NULL` дає `NULL` (не true і не false).

In [41]:
cursor.executemany(
    "INSERT INTO payments_demo (amount_uah, status, promo_code) VALUES (?, ?, ?)",
    [
        (1200, 'paid',     'SALE10'),
        (2000, 'paid',     None),      # промокоду немає (NULL)
        ( 800, 'refunded', ''),        # промокод — порожній рядок (це НЕ NULL)
        (3000, 'paid',     'BLACKFRIDAY'),
    ],
)
conn.commit()
sql("SELECT id, amount_uah, status, promo_code FROM payments_demo")

,id,amount_uah,status,promo_code
0,1,1500,pending,None
1,2,1200,paid,SALE10
2,3,2000,paid,None
3,4,800,refunded,
4,5,3000,paid,BLACKFRIDAY


In [42]:
sql("""
SELECT
    NULL = NULL      AS null_eq_null,
    NULL IS NULL     AS null_is_null,
    NULL + 10        AS null_plus_10,
    'abc' || NULL    AS concat_null,
    NULL = 0         AS null_eq_zero
""")

,null_eq_null,null_is_null,null_plus_10,concat_null,null_eq_zero
0,None,1,None,None,None


#### Пастка 1: `= NULL` не працює, потрібно `IS NULL`

In [43]:
sql("SELECT id, promo_code FROM payments_demo WHERE promo_code = NULL")   # завжди порожньо!

,id,promo_code


In [44]:
sql("SELECT id, promo_code FROM payments_demo WHERE promo_code IS NULL")

,id,promo_code
0,1,None
1,3,None


#### Пастка 2: рядки з NULL «зникають» з умов `<>`
Аналітик хоче порахувати оплати **без** промокоду `SALE10`:

In [45]:
sql("SELECT id, promo_code FROM payments_demo WHERE promo_code <> 'SALE10'")

,id,promo_code
0,4,
1,5,BLACKFRIDAY


Рядки з `NULL` не потрапили, бо `NULL <> 'SALE10'` → `NULL` → рядок відкидається. Правильно:

In [46]:
sql("""
SELECT id, promo_code
FROM payments_demo
WHERE promo_code <> 'SALE10' OR promo_code IS NULL
""")

,id,promo_code
0,1,None
1,3,None
2,4,
3,5,BLACKFRIDAY


#### `COALESCE` — підставити значення замість NULL
Аналог `fillna()` у pandas.

In [47]:
sql("""
SELECT id, promo_code,
       COALESCE(promo_code, 'без промокоду') AS promo_filled
FROM payments_demo
""")

,id,promo_code,promo_filled
0,1,None,без промокоду
1,2,SALE10,SALE10
2,3,None,без промокоду
3,4,,
4,5,BLACKFRIDAY,BLACKFRIDAY


#### Інші важливі моменти
- `COUNT(*)` рахує рядки, а `COUNT(стовпець)` — тільки **не-NULL** значення (детально — в уроці про агрегації).
- `UNIQUE` дозволяє **кілька NULL** — бо невідоме не дорівнює невідомому.
- При читанні в pandas `NULL` перетворюється на `None` / `NaN`.

In [48]:
sql("SELECT COUNT(*) AS all_rows, COUNT(promo_code) AS with_promo FROM payments_demo")

,all_rows,with_promo
0,5,3


In [49]:
df = sql("SELECT id, promo_code FROM payments_demo")
df['promo_code'].isna()

,promo_code
0,True
1,False
2,True
3,False
4,False


In [50]:
run("DROP TABLE types_demo")
run("DROP TABLE types_strict")
run("DROP TABLE payments_demo")

## 6. Перепроєктуємо базу школи

### 6.1. Нормалізуємо таблицю `school_flat` за шпаргалкою
Повертаємося до «плоскої» таблиці з розділу 2 і проходимо по ній трьома питаннями з розділу 3.5.

In [51]:
flat = pd.DataFrame(flat_rows, columns=[
    'student_name', 'student_email', 'student_phones', 'group_title',
    'course_title', 'course_hours', 'teacher_name', 'teacher_email',
])
flat

,student_name,student_email,student_phones,group_title,course_title,course_hours,teacher_name,teacher_email
0,Олена Коваль,olena@gmail.com,"050-111-11-11, 067-222-22-22",DA-01,Аналітика даних,120,Петро Іваненко,petro@school.ua
1,Олена Коваль,olena@gmail.com,"050-111-11-11, 067-222-22-22",PY-01,Python,80,Марія Литвин,maria@school.ua
2,Андрій Мельник,andrii@gmail.com,063-333-33-33,DA-01,Аналітика даних,120,Петро Іваненко,petro@school.ua
3,Ірина Шевченко,iryna@gmail.com,"050-444-44-44, 093-555-55-55",DA-02,Аналітика даних,120,Петро Іваненко,petro@school.ua
4,Ірина Шевченко,iryna@gmail.com,"050-444-44-44, 093-555-55-55",PY-01,Python,80,Марія Литвин,maria@school.ua
5,Максим Бондар,maksym@gmail.com,066-666-66-66,DA-02,Аналітика даних,120,Петро Іваненко,petro@school.ua
6,Софія Ткаченко,sofia@gmail.com,097-777-77-77,ML-01,Machine Learning,100,Олег Савчук,oleg@school.ua


#### Питання 1: у клітинці список? → 1НФ
Так: `student_phones`. Виносимо телефони в окрему таблицю, по одному в рядку (як у прикладі 3.1).

In [52]:
student_phones = (
    flat[['student_email', 'student_phones']]
    .drop_duplicates()
    .assign(phone=lambda d: d['student_phones'].str.split(', '))
    .explode('phone')
    .drop(columns='student_phones')
    .reset_index(drop=True)
)
student_phones

,student_email,phone
0,olena@gmail.com,050-111-11-11
1,olena@gmail.com,067-222-22-22
2,andrii@gmail.com,063-333-33-33
3,iryna@gmail.com,050-444-44-44
4,iryna@gmail.com,093-555-55-55
5,maksym@gmail.com,066-666-66-66
6,sofia@gmail.com,097-777-77-77


In [53]:
enroll = flat.drop(columns='student_phones')   # решта таблиці без телефонів

#### Який ключ?
Кожен рядок — це «студент навчається в групі». Один студент буває в кількох групах, у групі кілька студентів, тож ключ складений: **(`student_email`, `group_title`)**, як (студент, предмет) у прикладі 3.2.

In [54]:
print('Повтори за студентом:        ', enroll.duplicated(['student_email']).any())
print('Повтори за групою:           ', enroll.duplicated(['group_title']).any())
print('Повтори за (студент, група): ', enroll.duplicated(['student_email', 'group_title']).any())

Повтори за студентом:         True
Повтори за групою:            True
Повтори за (студент, група):  False


#### Питання 2: стовпець залежить лише від частини ключа? → 2НФ
| Стовпець | Залежить від | |
|---|---|---|
| `student_name` | лише `student_email` | ❌ частина ключа |
| `course_title`, `course_hours`, `teacher_name`, `teacher_email` | лише `group_title` | ❌ частина ключа |

Як і з містом у прикладі 3.3, виносимо кожну групу стовпців у таблицю, де ключ — ця частина:

In [55]:
students = enroll[['student_email', 'student_name']].drop_duplicates().reset_index(drop=True)
students

,student_email,student_name
0,olena@gmail.com,Олена Коваль
1,andrii@gmail.com,Андрій Мельник
2,iryna@gmail.com,Ірина Шевченко
3,maksym@gmail.com,Максим Бондар
4,sofia@gmail.com,Софія Ткаченко


In [56]:
groups_2nf = (
    enroll[['group_title', 'course_title', 'course_hours', 'teacher_name', 'teacher_email']]
    .drop_duplicates()
    .reset_index(drop=True)
)
groups_2nf

,group_title,course_title,course_hours,teacher_name,teacher_email
0,DA-01,Аналітика даних,120,Петро Іваненко,petro@school.ua
1,PY-01,Python,80,Марія Литвин,maria@school.ua
2,DA-02,Аналітика даних,120,Петро Іваненко,petro@school.ua
3,ML-01,Machine Learning,100,Олег Савчук,oleg@school.ua


In [57]:
students_in_groups = enroll[['student_email', 'group_title']]   # лишився лише сам факт "хто в якій групі"
students_in_groups

,student_email,group_title
0,olena@gmail.com,DA-01
1,olena@gmail.com,PY-01
2,andrii@gmail.com,DA-01
3,iryna@gmail.com,DA-02
4,iryna@gmail.com,PY-01
5,maksym@gmail.com,DA-02
6,sofia@gmail.com,ML-01


#### Питання 3: стовпець описує інший неключовий стовпець? → 3НФ
У `groups_2nf` ключ — `group_title`. Але:
- `course_hours` — властивість **курсу**, а не групи (група → курс → тривалість);
- `teacher_name` — властивість **викладача**, а не групи (група → викладач → ім'я).

Як із куратором у прикладі 3.4, виносимо курси й викладачів:

In [58]:
courses = groups_2nf[['course_title', 'course_hours']].drop_duplicates().reset_index(drop=True)
courses

,course_title,course_hours
0,Аналітика даних,120
1,Python,80
2,Machine Learning,100


In [59]:
teachers = groups_2nf[['teacher_email', 'teacher_name']].drop_duplicates().reset_index(drop=True)
teachers

,teacher_email,teacher_name
0,petro@school.ua,Петро Іваненко
1,maria@school.ua,Марія Литвин
2,oleg@school.ua,Олег Савчук


In [60]:
study_groups = groups_2nf[['group_title', 'course_title', 'teacher_email']]
study_groups

,group_title,course_title,teacher_email
0,DA-01,Аналітика даних,petro@school.ua
1,PY-01,Python,maria@school.ua
2,DA-02,Аналітика даних,petro@school.ua
3,ML-01,Machine Learning,oleg@school.ua


#### Перевіряємо аномалії з розділу 2
| Аномалія | Було | Стало |
|---|---|---|
| Оновлення email Петра | змінювати 4 рядки, легко пропустити | змінити **1 рядок** у `teachers` |
| Пошук за телефоном | `LIKE` по склеєному рядку | точний пошук у `student_phones` |
| Новий курс без студентів | «рядок-привид» | звичайний рядок у `courses` |
| Пішла єдина студентка курсу | зник курс і викладач | видаляємо лише рядок у `students_in_groups` |

#### Отримана схема
```
 teachers                 courses
 ────────                 ───────
 teacher ◄──┐        ┌──► course
            │        │
         study_groups (група: курс + викладач)
                 ▲
                 │
        students_in_groups (студент ↔ група, зв'язок M:N)
                 │
                 ▼
             students ◄──── student_phones (зв'язок 1:N)
```

#### Від email до `id`
У pandas ми зв'язували таблиці через email і назви. У базі, як обговорили в розділі 4, кожна таблиця отримає **сурогатний ключ `id`**, а email і назви стануть `UNIQUE`. Зв'язки будуть через **зовнішні ключі** на `id`.

### 6.2. Що виправляємо порівняно з першою версією схеми
| # | Було | Проблема | Рішення |
|---|---|---|---|
| 1 | немає `PRAGMA foreign_keys = ON` | зовнішні ключі в SQLite **не перевіряються** | вмикаємо після кожного `connect()` |
| 2 | немає `conn.commit()` | зміни можуть не потрапити у файл | `commit()` після змін / `with conn:` |
| 3 | `students_in_groups` з окремим `id` і без `UNIQUE` | можна записати студента в одну групу **двічі** | складений PK `(student_id, group_id)` |
| 4 | у `groups` немає викладача | `teachers` ні з чим не пов'язана | `study_groups.teacher_id → teachers.id` |
| 5 | таблиця `groups` | `GROUPS` — ключове слово SQL (у MySQL 8 зарезервоване; в SQLite працює лише завдяки поблажливості парсера) | назва `study_groups` |
| 6 | `ALTER TABLE ... ADD COLUMN phone_number TEXT NOT NULL UNIQUE` | падає в SQLite (див. розділ ALTER) + у студента може бути кілька телефонів (1НФ) | окрема таблиця `student_phones` |
| 7 | немає політики `ON DELETE` | незрозуміло, що буде з групами при видаленні курсу | задаємо явно |
| 8 | немає `CHECK` / `DEFAULT` | можна записати `hours = -5`, дату `'вчора'` | `CHECK`, `DEFAULT`, `is_active`, `created_at` |
| 9 | назви курсів/груп не унікальні | два курси «Python» | `UNIQUE` |
| 10 | `AUTOINCREMENT` | не помилка, але зазвичай зайве | достатньо `INTEGER PRIMARY KEY` |

### 6.3. Порядок створення
Спочатку **батьківські** таблиці (на які посилаються), потім **дочірні**: `teachers`, `courses` → `study_groups` → `students` → `student_phones`, `students_in_groups`.

Синтаксис:
```sql
CREATE TABLE [IF NOT EXISTS] назва (
    стовпець ТИП [обмеження],
    ...,
    [обмеження рівня таблиці]
);
```
`IF NOT EXISTS` — не падати, якщо таблиця вже є (корисно при повторному запуску).

In [61]:
cursor.execute("PRAGMA foreign_keys = ON")   # ще раз: обов'язково для кожного підключення!

run("""
CREATE TABLE IF NOT EXISTS teachers (
    id          INTEGER PRIMARY KEY,
    first_name  TEXT    NOT NULL,
    last_name   TEXT    NOT NULL,
    email       TEXT    NOT NULL UNIQUE CHECK (email LIKE '%_@_%._%'),
    mentor_id   INTEGER REFERENCES teachers(id) ON DELETE SET NULL,   -- посилання на себе
    hired_at    TEXT    NOT NULL DEFAULT CURRENT_DATE
                        CHECK (hired_at = date(hired_at))             -- тільки коректна дата YYYY-MM-DD
)
""")

In [62]:
run("""
CREATE TABLE IF NOT EXISTS courses (
    id     INTEGER PRIMARY KEY,
    title  TEXT    NOT NULL UNIQUE,
    hours  INTEGER NOT NULL CHECK (hours > 0)
)
""")

In [63]:
run("""
CREATE TABLE IF NOT EXISTS study_groups (
    id          INTEGER PRIMARY KEY,
    title       TEXT    NOT NULL UNIQUE,
    course_id   INTEGER NOT NULL REFERENCES courses(id)  ON DELETE RESTRICT,  -- курс з групами не видалити
    teacher_id  INTEGER          REFERENCES teachers(id) ON DELETE SET NULL,  -- викладача можуть ще не призначити
    start_date  TEXT    CHECK (start_date = date(start_date))
)
""")

Про `date_of_birth`: можна писати тип `DATE` — для читабельності. SQLite все одно збереже текст (див. розділ 5). Ми пишемо `TEXT` і перевіряємо формат через `CHECK`: `date(x)` повертає ту саму дату, лише якщо вона коректна.

In [64]:
run("""
CREATE TABLE IF NOT EXISTS students (
    id             INTEGER PRIMARY KEY,
    first_name     TEXT    NOT NULL,
    last_name      TEXT    NOT NULL,
    date_of_birth  TEXT    NOT NULL CHECK (date_of_birth = date(date_of_birth)),
    email          TEXT    NOT NULL UNIQUE CHECK (email LIKE '%_@_%._%'),
    is_active      INTEGER NOT NULL DEFAULT 1 CHECK (is_active IN (0, 1)),
    created_at     TEXT    NOT NULL DEFAULT CURRENT_TIMESTAMP
)
""")

In [65]:
run("""
CREATE TABLE IF NOT EXISTS student_phones (
    student_id  INTEGER NOT NULL REFERENCES students(id) ON DELETE CASCADE,
    phone       TEXT    NOT NULL,
    PRIMARY KEY (student_id, phone)        -- складений ключ: один номер у студента лише раз
)
""")

In [66]:
run("""
CREATE TABLE IF NOT EXISTS students_in_groups (
    student_id   INTEGER NOT NULL REFERENCES students(id)     ON DELETE CASCADE,
    group_id     INTEGER NOT NULL REFERENCES study_groups(id) ON DELETE CASCADE,
    enrolled_at  TEXT    NOT NULL DEFAULT CURRENT_DATE,
    PRIMARY KEY (student_id, group_id)     -- не можна записати студента в групу двічі
)
""")

### 6.4. Як подивитися структуру бази
- `sqlite_master` — службова таблиця зі списком усіх об'єктів бази;
- `PRAGMA table_info(таблиця)` — стовпці, типи, `NOT NULL`, `DEFAULT`, PK;
- `PRAGMA foreign_key_list(таблиця)` — зовнішні ключі.

In [67]:
sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,teachers
1,courses
2,study_groups
3,students
4,student_phones
5,students_in_groups


In [68]:
sql("PRAGMA table_info(students)")

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,first_name,TEXT,1,None,0
2,2,last_name,TEXT,1,None,0
3,3,date_of_birth,TEXT,1,None,0
4,4,email,TEXT,1,None,0
5,5,is_active,INTEGER,1,1,0
6,6,created_at,TEXT,1,CURRENT_TIMESTAMP,0


In [69]:
sql("PRAGMA foreign_key_list(students_in_groups)")

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,study_groups,group_id,id,NO ACTION,CASCADE,NONE
1,1,0,students,student_id,id,NO ACTION,CASCADE,NONE


### 6.5. Зміна структури: `ALTER TABLE`
У SQLite `ALTER TABLE` вміє небагато:
- `ADD COLUMN` — додати стовпець;
- `RENAME COLUMN` — перейменувати стовпець;
- `RENAME TO` — перейменувати таблицю;
- `DROP COLUMN` — видалити стовпець (з версії 3.35).

Змінити тип чи обмеження існуючого стовпця не можна — тільки перестворити таблицю. В PostgreSQL можливостей значно більше.

In [70]:
# демо на окремій таблиці з даними
run("CREATE TABLE alter_demo (id INTEGER PRIMARY KEY, name TEXT NOT NULL)")
run("INSERT INTO alter_demo (name) VALUES ('Олена'), ('Андрій')")

try_sql("ALTER TABLE alter_demo ADD COLUMN telegram TEXT")                              # ✅ нові клітинки = NULL
try_sql("ALTER TABLE alter_demo ADD COLUMN city TEXT NOT NULL")                         # ❌ а що записати в існуючі рядки?
try_sql("ALTER TABLE alter_demo ADD COLUMN city TEXT NOT NULL DEFAULT 'не вказано'")    # ✅ з DEFAULT можна
try_sql("ALTER TABLE alter_demo ADD COLUMN passport TEXT UNIQUE")                       # ❌ UNIQUE через ALTER у SQLite не можна

sql("SELECT * FROM alter_demo")

✅ Виконано.
❌ OperationalError: Cannot add a NOT NULL column with default value NULL
✅ Виконано.
❌ OperationalError: Cannot add a UNIQUE column


,id,name,telegram,city
0,1,Олена,None,не вказано
1,2,Андрій,None,не вказано


Саме тому запит `ADD COLUMN phone_number TEXT NOT NULL UNIQUE` з першої версії завершується помилкою: `UNIQUE` додати через `ALTER` не можна ніколи, а `NOT NULL` без `DEFAULT` — якщо в таблиці вже є рядки (на порожній таблиці новіші версії SQLite це дозволяють).

Унікальність для нового стовпця можна задати окремо через унікальний індекс: `CREATE UNIQUE INDEX ... ON таблиця(стовпець)` (індекси — в одному з наступних уроків).

In [71]:
try_sql("ALTER TABLE alter_demo RENAME COLUMN telegram TO telegram_nick")
try_sql("ALTER TABLE alter_demo DROP COLUMN city")
try_sql("ALTER TABLE alter_demo RENAME TO alter_demo_renamed")

sql("PRAGMA table_info(alter_demo_renamed)")[['name', 'type', 'notnull', 'dflt_value']]

✅ Виконано.
✅ Виконано.
✅ Виконано.


,name,type,notnull,dflt_value
0,id,INTEGER,0,None
1,name,TEXT,1,None
2,telegram_nick,TEXT,0,None


### 6.6. Видалення таблиці: `DROP TABLE`
```sql
DROP TABLE [IF EXISTS] назва;
```
Видаляє таблицю **разом з даними**, без підтвердження і без «кошика». Якщо на таблицю посилаються інші (і FK увімкнені), видалити її з даними не вийде.

In [72]:
run("DROP TABLE IF EXISTS alter_demo_renamed")
try_sql("DROP TABLE alter_demo_renamed")             # вдруге — помилка: таблиці вже немає
try_sql("DROP TABLE IF EXISTS alter_demo_renamed")   # а з IF EXISTS — ні

❌ OperationalError: no such table: alter_demo_renamed
✅ Виконано.


## 7. Чотири типи запитів до даних: CRUD

| Операція | SQL | Що робить |
|---|---|---|
| **C**reate | `INSERT` | додати рядки |
| **R**ead | `SELECT` | прочитати |
| **U**pdate | `UPDATE` | змінити існуючі |
| **D**elete | `DELETE` | видалити |

### INSERT — додавання даних
```sql
INSERT INTO таблиця (стовпець1, стовпець2, ...)
VALUES (значення1, значення2, ...);
```
Завжди **вказуйте список стовпців**: запит не зламається, якщо в таблицю додадуть новий стовпець або змінять їх порядок, а пропущені стовпці отримають `DEFAULT`.

In [73]:
# один рядок
run("INSERT INTO courses (title, hours) VALUES ('Аналітика даних', 120)")

# кілька рядків одним запитом
run("""
INSERT INTO courses (title, hours) VALUES
    ('Python', 80),
    ('Machine Learning', 100)
""")

sql("SELECT * FROM courses")

,id,title,hours
0,1,Аналітика даних,120
1,2,Python,80
2,3,Machine Learning,100


### Параметри `?` замість f-рядків
Коли значення приходять ззовні (з форми, файлу, від користувача), **ніколи** не вставляйте їх у запит через f-рядок. Використовуйте параметри `?` (або іменовані `:name`) — драйвер сам безпечно підставить значення.

`executemany` виконує один запит для списку значень. `cursor.lastrowid` — `id` щойно доданого рядка.

In [74]:
teachers_data = [
    ('Петро', 'Іваненко', 'petro@school.ua', None, '2020-09-01'),
    ('Марія', 'Литвин',   'maria@school.ua', 1,    '2022-02-14'),
    ('Олег',  'Савчук',   'oleg@school.ua',  1,    '2023-08-21'),
]

cursor.executemany(
    """
    INSERT INTO teachers (first_name, last_name, email, mentor_id, hired_at)
    VALUES (?, ?, ?, ?, ?)
    """,
    teachers_data,
)
conn.commit()

sql("SELECT * FROM teachers")

,id,first_name,last_name,email,mentor_id,hired_at
0,1,Петро,Іваненко,petro@school.ua,NaN,2020-09-01
1,2,Марія,Литвин,maria@school.ua,1.0,2022-02-14
2,3,Олег,Савчук,oleg@school.ua,1.0,2023-08-21


In [75]:
# іменовані параметри
cursor.execute(
    "INSERT INTO teachers (first_name, last_name, email, mentor_id) VALUES (:fn, :ln, :email, :mentor)",
    {'fn': 'Анна', 'ln': 'Гнатюк', 'email': 'anna@school.ua', 'mentor': 2},
)
conn.commit()
print('id нового викладача:', cursor.lastrowid)   # hired_at заповнився через DEFAULT CURRENT_DATE
sql("SELECT * FROM teachers")

id нового викладача: 4


,id,first_name,last_name,email,mentor_id,hired_at
0,1,Петро,Іваненко,petro@school.ua,NaN,2020-09-01
1,2,Марія,Литвин,maria@school.ua,1.0,2022-02-14
2,3,Олег,Савчук,oleg@school.ua,1.0,2023-08-21
3,4,Анна,Гнатюк,anna@school.ua,2.0,2026-11-04


In [76]:
cursor.executemany(
    "INSERT INTO study_groups (title, course_id, teacher_id, start_date) VALUES (?, ?, ?, ?)",
    [
        ('DA-01', 1, 1, '2026-09-01'),
        ('DA-02', 1, 1, '2026-10-01'),
        ('PY-01', 2, 2, '2026-09-15'),
        ('ML-01', 3, 3, '2026-10-10'),
    ],
)

cursor.executemany(
    "INSERT INTO students (first_name, last_name, date_of_birth, email) VALUES (?, ?, ?, ?)",
    [
        ('Олена',  'Коваль',   '2001-05-12', 'olena@gmail.com'),
        ('Андрій', 'Мельник',  '1998-11-30', 'andrii@gmail.com'),
        ('Ірина',  'Шевченко', '2003-02-07', 'iryna@gmail.com'),
        ('Максим', 'Бондар',   '1995-07-19', 'maksym@gmail.com'),
        ('Софія',  'Ткаченко', '2000-12-01', 'sofia@gmail.com'),
    ],
)

cursor.executemany(
    "INSERT INTO student_phones (student_id, phone) VALUES (?, ?)",
    [
        (1, '050-111-11-11'), (1, '067-222-22-22'),
        (2, '063-333-33-33'),
        (3, '050-444-44-44'), (3, '093-555-55-55'),
        (4, '066-666-66-66'),
        (5, '097-777-77-77'),
    ],
)

cursor.executemany(
    "INSERT INTO students_in_groups (student_id, group_id) VALUES (?, ?)",
    [(1, 1), (1, 3), (2, 1), (3, 2), (3, 3), (4, 2), (5, 4)],
)

conn.commit()
sql("SELECT * FROM students")

,id,first_name,last_name,date_of_birth,email,is_active,created_at
0,1,Олена,Коваль,2001-05-12,olena@gmail.com,1,2026-11-04 15:20:43
1,2,Андрій,Мельник,1998-11-30,andrii@gmail.com,1,2026-11-04 15:20:43
2,3,Ірина,Шевченко,2003-02-07,iryna@gmail.com,1,2026-11-04 15:20:43
3,4,Максим,Бондар,1995-07-19,maksym@gmail.com,1,2026-11-04 15:20:43
4,5,Софія,Ткаченко,2000-12-01,sofia@gmail.com,1,2026-11-04 15:20:43


Тепер база сама захищає дані — перевіримо:

In [77]:
try_sql("INSERT INTO students_in_groups (student_id, group_id) VALUES (1, 1)")     # Олена вже в DA-01
try_sql("INSERT INTO students_in_groups (student_id, group_id) VALUES (1, 99)")    # групи 99 немає
try_sql("INSERT INTO students (first_name, last_name, date_of_birth, email) "
        "VALUES ('Тест', 'Тестенко', '2001-02-30', 'test@gmail.com')")              # 30 лютого
try_sql("INSERT INTO students (first_name, last_name, date_of_birth, email) "
        "VALUES ('Тест', 'Тестенко', '2001-02-03', 'olena@gmail.com')")             # email зайнятий
try_sql("INSERT INTO courses (title, hours) VALUES ('Python', 40)")                  # курс уже є

❌ IntegrityError: UNIQUE constraint failed: students_in_groups.student_id, students_in_groups.group_id
❌ IntegrityError: FOREIGN KEY constraint failed
❌ IntegrityError: CHECK constraint failed: date_of_birth = date(date_of_birth)
❌ IntegrityError: UNIQUE constraint failed: students.email
❌ IntegrityError: UNIQUE constraint failed: courses.title


#### SQL-ін'єкція: чому f-рядки небезпечні
Уявімо форму «знайти студента за email», і користувач ввів не email, а ось це:

In [78]:
user_input = "' OR '1'='1"

bad_query = f"SELECT first_name, last_name, email FROM students WHERE email = '{user_input}'"
print(bad_query)
sql(bad_query)        # умова завжди істинна -> отримали ВСІХ студентів

SELECT first_name, last_name, email FROM students WHERE email = '' OR '1'='1'


,first_name,last_name,email
0,Олена,Коваль,olena@gmail.com
1,Андрій,Мельник,andrii@gmail.com
2,Ірина,Шевченко,iryna@gmail.com
3,Максим,Бондар,maksym@gmail.com
4,Софія,Ткаченко,sofia@gmail.com


In [79]:
# з параметром введення сприймається як звичайний рядок
sql("SELECT first_name, last_name, email FROM students WHERE email = ?", (user_input,))

,first_name,last_name,email


### SELECT — читання даних (базово)
```sql
SELECT стовпці
FROM таблиця
WHERE умова
ORDER BY стовпець [ASC | DESC]
LIMIT n;
```
Детально `SELECT` розберемо в наступних уроках, зараз — мінімум.

In [80]:
sql("SELECT * FROM courses")     # * — усі стовпці

,id,title,hours
0,1,Аналітика даних,120
1,2,Python,80
2,3,Machine Learning,100


In [81]:
# конкретні стовпці + псевдоніми (AS)
sql("SELECT first_name AS name, last_name AS surname, email FROM students")

,name,surname,email
0,Олена,Коваль,olena@gmail.com
1,Андрій,Мельник,andrii@gmail.com
2,Ірина,Шевченко,iryna@gmail.com
3,Максим,Бондар,maksym@gmail.com
4,Софія,Ткаченко,sofia@gmail.com


In [82]:
# фільтр, сортування, обмеження кількості
sql("""
SELECT first_name, last_name, date_of_birth
FROM students
WHERE date_of_birth >= '2000-01-01'
ORDER BY date_of_birth DESC
LIMIT 3
""")

,first_name,last_name,date_of_birth
0,Ірина,Шевченко,2003-02-07
1,Олена,Коваль,2001-05-12
2,Софія,Ткаченко,2000-12-01


In [83]:
# основні оператори в WHERE: =, <>, >, <, BETWEEN, IN, LIKE, IS NULL, AND, OR, NOT
sql("""
SELECT title, hours
FROM courses
WHERE hours BETWEEN 80 AND 110
   OR title LIKE '%Анал%'
""")

,title,hours
0,Аналітика даних,120
1,Python,80
2,Machine Learning,100


In [84]:
# DISTINCT — унікальні значення
sql("SELECT DISTINCT course_id FROM study_groups")

,course_id
0,1
1,2
2,3


### UPDATE — зміна даних
```sql
UPDATE таблиця
SET стовпець1 = значення1, стовпець2 = значення2
WHERE умова;
```
Пам'ятаєте аномалію оновлення email Петра? Тепер email зберігається **в одному місці**:

In [85]:
run("UPDATE teachers SET email = 'p.ivanenko@school.ua' WHERE id = 1")
sql("SELECT id, first_name, last_name, email FROM teachers")

,id,first_name,last_name,email
0,1,Петро,Іваненко,p.ivanenko@school.ua
1,2,Марія,Литвин,maria@school.ua
2,3,Олег,Савчук,oleg@school.ua
3,4,Анна,Гнатюк,anna@school.ua


In [86]:
# кілька стовпців одночасно; нове значення може обчислюватись зі старого
run("UPDATE courses SET hours = hours + 10, title = 'Python для аналітиків' WHERE id = 2")
sql("SELECT * FROM courses")

,id,title,hours
0,1,Аналітика даних,120
1,2,Python для аналітиків,90
2,3,Machine Learning,100


#### ⚠️ UPDATE без WHERE змінює **всі** рядки
Подивимось на це безпечно: виконаємо запит **без** `commit`, а потім відкотимо зміни через `rollback()`.

In [87]:
cursor.execute("UPDATE courses SET hours = 1")   # забули WHERE!
sql("SELECT * FROM courses")

,id,title,hours
0,1,Аналітика даних,1
1,2,Python для аналітиків,1
2,3,Machine Learning,1


In [88]:
conn.rollback()   # скасовуємо незафіксовані зміни
sql("SELECT * FROM courses")

,id,title,hours
0,1,Аналітика даних,120
1,2,Python для аналітиків,90
2,3,Machine Learning,100


> **Звичка:** спочатку напишіть `SELECT ... WHERE <умова>` і подивіться, які рядки потрапляють під умову. Тільки потім замініть `SELECT` на `UPDATE` / `DELETE` з **тим самим** `WHERE`.

### DELETE — видалення даних
```sql
DELETE FROM таблиця WHERE умова;
```
Пам'ятаєте аномалію видалення? Софія йде зі школи:

In [89]:
run("DELETE FROM students WHERE email = 'sofia@gmail.com'")

# її записи в групи й телефони видалились каскадно (ON DELETE CASCADE)
sql("SELECT * FROM students_in_groups WHERE student_id = 5")

,student_id,group_id,enrolled_at


In [90]:
# а курс і викладач на місці — аномалії більше немає
sql("SELECT * FROM courses")

,id,title,hours
0,1,Аналітика даних,120
1,2,Python для аналітиків,90
2,3,Machine Learning,100


In [91]:
# курс, у якого є групи, видалити не можна (ON DELETE RESTRICT)
try_sql("DELETE FROM courses WHERE id = 1")

❌ IntegrityError: FOREIGN KEY constraint failed


In [92]:
# викладача видалити можна: у його групах teacher_id стане NULL, у підопічних mentor_id стане NULL
run("DELETE FROM teachers WHERE id = 3")
sql("SELECT * FROM study_groups")

,id,title,course_id,teacher_id,start_date
0,1,DA-01,1,1.0,2026-09-01
1,2,DA-02,1,1.0,2026-10-01
2,3,PY-01,2,2.0,2026-09-15
3,4,ML-01,3,NaN,2026-10-10


#### М'яке видалення (soft delete)
У продукті користувачів часто **не видаляють фізично**, а позначають прапорцем (`is_active = 0`, `deleted_at = ...`). Для аналітика це дуже важливо: зберігається історія, і можна порахувати відтік (churn). Але тоді у запитах треба пам'ятати про фільтр `WHERE is_active = 1`.

In [93]:
run("UPDATE students SET is_active = 0 WHERE email = 'maksym@gmail.com'")
sql("SELECT first_name, last_name, is_active FROM students")

,first_name,last_name,is_active
0,Олена,Коваль,1
1,Андрій,Мельник,1
2,Ірина,Шевченко,1
3,Максим,Бондар,0


### Транзакції: `commit` і `rollback`
**Транзакція** — група операцій, що виконується **повністю або ніяк**. Модуль `sqlite3` автоматично відкриває транзакцію перед `INSERT`/`UPDATE`/`DELETE`:
- `conn.commit()` — зафіксувати зміни;
- `conn.rollback()` — скасувати все з моменту останнього `commit`.

Зручно використовувати `with conn:` — при успіху буде `commit`, при помилці — `rollback`.

Приклад: записуємо нову студентку і одразу в групу. Друга операція падає (групи 99 немає) — і першу теж скасовано:

In [94]:
try:
    with conn:
        conn.execute(
            "INSERT INTO students (first_name, last_name, date_of_birth, email) VALUES (?, ?, ?, ?)",
            ('Дарина', 'Кравець', '2002-04-15', 'daryna@gmail.com'),
        )
        conn.execute(
            "INSERT INTO students_in_groups (student_id, group_id) VALUES (last_insert_rowid(), 99)"
        )
except sqlite3.IntegrityError as e:
    print('❌ Помилка, транзакцію скасовано:', e)

sql("SELECT * FROM students WHERE email = 'daryna@gmail.com'")   # порожньо — студентку не додано

❌ Помилка, транзакцію скасовано: FOREIGN KEY constraint failed


,id,first_name,last_name,date_of_birth,email,is_active,created_at


### Зв'язок з pandas
- `pd.read_sql_query(query, conn)` — результат запиту в `DataFrame` (ми весь урок це робили);
- `df.to_sql('таблиця', conn, index=False, if_exists='replace' | 'append' | 'fail')` — записати `DataFrame` у базу.

`to_sql` зручний для швидкого завантаження CSV у базу, але створює таблицю **без** ключів і обмежень. Для «справжньої» таблиці краще спочатку `CREATE TABLE`, а потім `if_exists='append'`.

In [95]:
courses.to_sql('courses_from_pandas', conn, index=False, if_exists='replace')
sql("SELECT * FROM courses_from_pandas")

,course_title,course_hours
0,Аналітика даних,120
1,Python,80
2,Machine Learning,100


In [96]:
sql("SELECT sql FROM sqlite_master WHERE name = 'courses_from_pandas'").iloc[0, 0]   # без PK, NOT NULL, CHECK

'CREATE TABLE "courses_from_pandas" (\n"course_title" TEXT,\n  "course_hours" INTEGER\n)'

In [97]:
run("DROP TABLE courses_from_pandas")

### Зберегти базу
Файл `school.db` можна завантажити й відкрити, наприклад, у [DB Browser for SQLite](https://sqlitebrowser.org/) або DBeaver.

In [98]:
conn.commit()

try:
    from google.colab import files
    files.download(DB_NAME)
except ImportError:
    print('Не Colab: файл бази лежить тут ->', os.path.abspath(DB_NAME))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Практика

Виконуйте запити через `run(...)` / `try_sql(...)` / `sql(...)`. Рішення сховані під спойлером.

**Завдання 1.** Додайте курс «SQL для аналітиків» тривалістю 60 годин.

<details><summary>Рішення</summary>

```sql
INSERT INTO courses (title, hours) VALUES ('SQL для аналітиків', 60);
```
</details>

**Завдання 2.** Створіть групу `SQL-01` на цьому курсі з викладачкою Марією Литвин, старт — 2026-11-16.

<details><summary>Рішення</summary>

```sql
INSERT INTO study_groups (title, course_id, teacher_id, start_date)
VALUES ('SQL-01', 4, 2, '2026-11-16');
```
`id` курсу і викладачки беремо з таблиць `courses` і `teachers`.
</details>

**Завдання 3.** Запишіть Андрія Мельника та Ірину Шевченко в групу `SQL-01`. Потім спробуйте записати Андрія вдруге і поясніть помилку.

<details><summary>Рішення</summary>

```sql
INSERT INTO students_in_groups (student_id, group_id) VALUES (2, 5), (3, 5);
INSERT INTO students_in_groups (student_id, group_id) VALUES (2, 5);  -- UNIQUE constraint failed
```
Пара `(student_id, group_id)` — первинний ключ, тому повтор заборонено.
</details>

**Завдання 4.** Виведіть ім'я, прізвище та дату народження студентів, народжених до 2001 року, від наймолодшого до найстаршого.

<details><summary>Рішення</summary>

```sql
SELECT first_name, last_name, date_of_birth
FROM students
WHERE date_of_birth < '2001-01-01'
ORDER BY date_of_birth DESC;
```
</details>

**Завдання 5.** Додайте Олені Коваль ще один телефон `073-888-88-88`. Потім виведіть усі її телефони.

<details><summary>Рішення</summary>

```sql
INSERT INTO student_phones (student_id, phone) VALUES (1, '073-888-88-88');
SELECT phone FROM student_phones WHERE student_id = 1;
```
</details>

**Завдання 6.** Курс «Machine Learning» скоротили до 90 годин. Оновіть дані. Перед цим перевірте через `SELECT`, що під умову потрапляє рівно один рядок.

<details><summary>Рішення</summary>

```sql
SELECT * FROM courses WHERE title = 'Machine Learning';
UPDATE courses SET hours = 90 WHERE title = 'Machine Learning';
```
</details>

**Завдання 7.** Спробуйте видалити курс «Аналітика даних». Чому не виходить? Що треба зробити, щоб видалення спрацювало?

<details><summary>Рішення</summary>

На курс посилаються групи `DA-01`, `DA-02`, а `study_groups.course_id` має `ON DELETE RESTRICT`. Спочатку треба видалити (або перенести на інший курс) ці групи. Записи студентів у групах видаляться каскадно.
</details>

**Завдання 8 (проєктування).** Спроєктуйте та створіть таблицю `payments` — оплати студентів:
- студент (зв'язок зі `students`), сума в гривнях (> 0, обов'язкова);
- статус: `pending` / `paid` / `refunded`, за замовчуванням `pending`;
- дата створення (за замовчуванням — поточний час), дата оплати (може бути порожньою).

Яку політику `ON DELETE` ви оберете для студента і чому?

<details><summary>Один з варіантів</summary>

```sql
CREATE TABLE IF NOT EXISTS payments (
    id          INTEGER PRIMARY KEY,
    student_id  INTEGER NOT NULL REFERENCES students(id) ON DELETE RESTRICT,
    amount_uah  NUMERIC NOT NULL CHECK (amount_uah > 0),
    status      TEXT    NOT NULL DEFAULT 'pending'
                        CHECK (status IN ('pending', 'paid', 'refunded')),
    created_at  TEXT    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    paid_at     TEXT
);
```
`RESTRICT`: фінансову історію не можна втрачати. Студентів, які пішли, краще деактивувати (`is_active = 0`), ніж видаляти.
</details>

**Завдання 9 (нормалізація, домашнє).** Інтернет-кав'ярня веде замовлення в одній таблиці:

| order_id | order_date | customer_name | customer_email | items | total |
|---|---|---|---|---|---|
| 1 | 2026-11-01 | Ігор | igor@gmail.com | Лате x2, Круасан x1 | 205 |
| 2 | 2026-11-01 | Оля | olia@gmail.com | Капучино x1 | 70 |
| 3 | 2026-11-02 | Ігор | igor@gmail.com | Круасан x2 | 130 |

Ціни: Лате — 70, Капучино — 70, Круасан — 65.

1. Які аномалії можливі в цій таблиці? Наведіть приклад кожної.
2. Приведіть схему до 3НФ (таблиці, стовпці, ключі, зв'язки).
3. Напишіть `CREATE TABLE` для всіх таблиць з `PRIMARY KEY`, `FOREIGN KEY`, `NOT NULL`, `CHECK`, `DEFAULT`, де доречно.
4. Чи потрібно зберігати стовпець `total`? А ціну товару в момент покупки? Обґрунтуйте.

## Підсумки
- **БД** зберігає дані надійно й узгоджено; **СУБД** — програма, що нею керує; **SQL** — мова запитів.
- Одна велика таблиця → дублювання й **аномалії** оновлення, вставки, видалення.
- **Нормалізація**: 1НФ — атомарні значення; 2НФ — немає залежностей від частини ключа; 3НФ — немає залежностей між неключовими стовпцями.
- **PK** ідентифікує рядок, **FK** зв'язує таблиці; **M:N** — через проміжну таблицю; **self-reference** — для ієрархій.
- У SQLite: типи — лише рекомендація (або `STRICT`), дати — рядки `YYYY-MM-DD`, **FK вмикаються через `PRAGMA foreign_keys = ON`**.
- **NULL** ≠ 0 ≠ `''`; порівнюємо через `IS NULL`; пам'ятаємо про `<>` і `COALESCE`.
- **CRUD**: `INSERT`, `SELECT`, `UPDATE`, `DELETE`; завжди `WHERE` в `UPDATE`/`DELETE`; параметри `?` замість f-рядків; `commit` / `rollback`.

**Далі:** `SELECT` детально, агрегації (`GROUP BY`), `JOIN`, підзапити, віконні функції — і продуктові метрики на SQL.

In [99]:
conn.close()